In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
from sodapy import Socrata
import geopandas as gpd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

BASE_DIR      = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = os.path.join(BASE_DIR, "Scripts Python", "webpage_climate", "data")

os.chdir(BASE_DIR)
print("Directorio de trabajo :", os.getcwd())
print("Carpeta web/data      :", WEB_DATA_DIR)

Directorio de trabajo : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data
Carpeta web/data      : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data


In [ ]:
pip install geopandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install sodapy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Datos diarios – Extracción API y cruce con alertas históricas

In [ ]:
# Carga alertas históricas generadas por datos precipitacion historicos.ipynb
out_estaciones = os.path.join(WEB_DATA_DIR, "alerta_historica_estaciones.csv")
estaciones_alerta = pd.read_csv(out_estaciones)

print(f"Estaciones históricas cargadas: {len(estaciones_alerta):,}")
print("\n=== Distribución de alerta compuesta (histórica) ===")
estaciones_alerta.head()

Estaciones históricas cargadas: 3,053

=== Distribución de alerta compuesta (histórica) ===


,CodigoEstacion,NombreEstacion,Departamento,Municipio,Latitud,Longitud,frecuencia_extremos,pendiente,p_valor,tendencia,frecuencia_reciente,ratio_reciente,alerta_lluvias,sequia_categoria,cod_norm
0,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,5.412000,-76.418000,0.100467,NaN,NaN,NaN,NaN,NaN,ALTA,NORMAL,11017020
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,5.888719,-76.145167,0.104966,NaN,NaN,NaN,NaN,NaN,ALTA,NORMAL,11025501
2,11027030,EL SIETE,CHOCO,EL CARMEN,5.862000,-76.152056,0.070225,-0.008545,0.153227,estable,0.096455,1.373513,MODERADA,MODERADA,11027030
3,11027030,EL SIETE - AUT,CHOCO,EL CARMEN,5.862000,-76.152056,0.178571,-0.008545,0.153227,estable,0.096455,0.540146,ALTA,MODERADA,11027030
4,11027070,BORAUDO,CHOCÓ,LLORÓ,5.515000,-76.576000,0.101568,NaN,NaN,NaN,NaN,NaN,ALTA,NORMAL,11027070


In [ ]:
# Carga davipola y proyecta a EPSG 3116 (necesario para sjoin con municipios)
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

print(f"Municipios cargados: {len(gdf_mun):,}")

Municipios cargados: 1,121


## Datos en tiempo real – API IDEAM

In [ ]:
DATASET_ID = "s54a-sgyg"
client = Socrata("www.datos.gov.co", None)
fecha_mapa = client.get(DATASET_ID, select="max(fechaobservacion)")[0]["max_fechaobservacion"][:10]
print(f"Última fecha disponible: {fecha_mapa}")

where = (
    f"fechaobservacion >= '{fecha_mapa}T00:00:00' "
    f"AND fechaobservacion < '{fecha_mapa}T23:59:59.999'"
)

records, offset = [], 0
while True:
    batch = client.get(DATASET_ID, where=where, limit=100_000, offset=offset)
    if not batch:
        break
    records.extend(batch)
    offset += 100_000
    print(f"  {len(records):,} registros descargados...")
client.close()

Última fecha disponible: 2026-03-17
  100,000 registros descargados...
  168,572 registros descargados...


In [ ]:
estaciones_alerta.columns

Index(['CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio',
       'Latitud', 'Longitud', 'frecuencia_extremos', 'pendiente', 'p_valor',
       'tendencia', 'frecuencia_reciente', 'ratio_reciente', 'alerta_lluvias',
       'sequia_categoria', 'cod_norm'],
      dtype='object')

In [ ]:
# Agrega lecturas a nivel de estación
df_api = pd.DataFrame.from_records(records)

for col in ("valorobservado", "latitud", "longitud"):
    df_api[col] = pd.to_numeric(df_api[col], errors="coerce")

df_api = df_api.dropna(subset=["latitud", "longitud", "valorobservado"])
df_api = df_api[df_api["valorobservado"] >= 0]

STATION_COLS_API = [
    "codigoestacion", "nombreestacion", "departamento",
    "municipio", "zonahidrografica", "latitud", "longitud",
]

df_dia = (
    df_api.groupby(STATION_COLS_API, as_index=False)
    .agg(
        precip_acum_mm=("valorobservado", "sum"),
        precip_max_10min=("valorobservado", "max"),
        n_lecturas=("valorobservado", "count"),
    )
)
del df_api

# Normaliza cod_norm a string en ambos lados antes del merge
df_dia["cod_norm"] = df_dia["codigoestacion"].astype(str).str.strip().str.lstrip("0")
estaciones_alerta["cod_norm"] = estaciones_alerta["cod_norm"].astype(str).str.strip().str.lstrip("0")

# Cruce con alertas históricas — incluye indicadores de sequía y tipo_alerta
cols_merge = [
    "cod_norm", 'alerta_lluvias',
    "frecuencia_extremos", "frecuencia_reciente", "ratio_reciente", "tendencia",
    "sequia_categoria"
]


df_dia = df_dia.merge(estaciones_alerta[cols_merge], on="cod_norm", how="left")

df_dia["alerta_lluvias"]= df_dia["alerta_lluvias"].fillna("BAJA")
df_dia["frecuencia_extremos"] = df_dia["frecuencia_extremos"].fillna(0)
df_dia["frecuencia_reciente"] = df_dia["frecuencia_reciente"].fillna(0)
df_dia["ratio_reciente"]      = df_dia["ratio_reciente"].fillna(np.nan)
df_dia["tendencia"]           = df_dia["tendencia"].fillna("sin_datos")
df_dia["sequia_categoria"]    = df_dia["sequia_categoria"].fillna("NORMAL")
#df_dia["tipo_alerta"]         = df_dia["tipo_alerta"].fillna("Sin alerta")

print(f"Estaciones activas hoy     : {len(df_dia):,}")
print(f"  Sin match en histórico    : {(df_dia['frecuencia_extremos'] == 0).sum():,}")

Estaciones activas hoy     : 2,218
  Sin match en histórico    : 829


In [ ]:
df_dia.head()

,codigoestacion,nombreestacion,departamento,municipio,zonahidrografica,latitud,longitud,precip_acum_mm,precip_max_10min,n_lecturas,cod_norm,alerta_lluvias,frecuencia_extremos,frecuencia_reciente,ratio_reciente,tendencia,sequia_categoria
0,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,0.0,0.0,144,11027030,MODERADA,0.070225,0.096455,1.373513,estable,MODERADA
1,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,0.0,0.0,144,11027030,ALTA,0.178571,0.096455,0.540146,estable,MODERADA
2,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,0.0,0.0,144,11027030,BAJA,0.000000,0.000000,NaN,insuficiente,NORMAL
3,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,0.0,0.0,144,11027030,BAJA,0.000000,0.000000,NaN,insuficiente,NORMAL
4,0011030010,CERTEGUI,CHOCO,CÉRTEGUI,ATRATO - DARIÉN,5.380,-76.610000,0.0,0.0,144,11030010,CRÍTICA,0.461538,0.495833,1.074306,creciente,MODERADA


In [ ]:
# ── Helpers robustos ante NaN ─────────────────────────────────────────────
def modo_seguro(serie, default='BAJA'):
    vc = serie.dropna().value_counts()
    return vc.idxmax() if not vc.empty else default

def tendencia_muni(serie):
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'creciente' in vals.values:
        return 'creciente'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'sin_datos'

def modo_sequia(serie):
    return modo_seguro(serie, default='NORMAL')

NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}

# Sjoin estaciones del día → municipios
gdf_dia = gpd.GeoDataFrame(
    df_dia,
    geometry=gpd.points_from_xy(df_dia.longitud, df_dia.latitud),
    crs="EPSG:4326",
).to_crs(epsg=3116)

df_muni = gpd.sjoin_nearest(gdf_mun, gdf_dia, how="left", distance_col="dist_m")

# Agrupación a nivel municipal
df_muni = (
    df_muni
    .groupby(["COD_MPIO", "NOM_MPIO", "NOM_DPTO", "LATITUD", "LONGITUD"], as_index=False)
    .agg(
        precip_acum_mm=("precip_acum_mm", "mean"),
        precip_max_10min=("precip_max_10min", "max"),
        n_estaciones=("codigoestacion", "count"),
        alerta_compuesta=("alerta_lluvias", modo_seguro),
        frecuencia_extremos=("frecuencia_extremos", "max"),
        frecuencia_reciente=("frecuencia_reciente", "max"),
        ratio_reciente=("ratio_reciente", "max"),
        tendencia=("tendencia", tendencia_muni),
        sequia_categoria=("sequia_categoria", modo_sequia),
    )
)

# alerta numérico derivado de alerta_compuesta (moda) → consistencia garantizada
df_muni["alerta"] = df_muni["alerta_compuesta"].map(NIVEL_NUMERICO).fillna(0).astype(int)
df_muni["fecha"]  = fecha_mapa

print(f"Municipios con datos   : {len(df_muni):,}")
print(f"\n=== Alerta compuesta por municipio (nivel dominante) ===")
print(df_muni['alerta_compuesta'].value_counts())
print(f"\nTotal con alerta > BAJA: {(df_muni['alerta'] > 0).sum():,}")

out_path = os.path.join(WEB_DATA_DIR, "datos_municipios.csv")
df_muni.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\nGuardado:", out_path)
df_muni.sort_values('alerta', ascending=False).head(5)

In [ ]:
df_muni[df_muni["COD_MPIO"]=="19824"].head()